# Day 46: Monitoring & Logging with LangSmith and WandB

Monitor your LLM calls, trace execution, and log metrics.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# LangSmith setup
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "day46_demo"

In [ ]:
# 1. LangSmith with LangChain
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Simple call – automatically traced
response = llm.invoke([HumanMessage(content="What is the capital of Japan?")])
print(response.content)

# View trace at https://smith.langchain.com/

In [ ]:
# 2. Add custom metadata and tags
from langchain.callbacks.tracers import LangChainTracer

tracer = LangChainTracer(project_name="day46_metadata")
response = llm.invoke(
    [HumanMessage(content="Explain quantum computing")],
    config={"callbacks": [tracer], "tags": ["science", "quantum"], "metadata": {"user_id": "123"}}
)
print("Traced with metadata")

In [ ]:
# 3. Weights & Biases logging
import wandb

# Initialize a run
wandb.init(project="my_capstone", name="day46_logging")

# Log prompt and response
prompt = "Write a haiku about AI."
response = llm.invoke([HumanMessage(content=prompt)]).content

wandb.log({
    "prompt": prompt,
    "response": response,
    "model": "gpt-3.5-turbo",
    "latency_seconds": 0.5,  # Replace with actual measured latency
    "tokens": 50             # Replace with token count
})

print("Logged to wandb")
wandb.finish()

In [ ]:
# 4. Evaluate RAG pipeline with LangSmith (optional)
# from langsmith import Client
# client = Client()
# dataset = client.create_dataset("my_qa_pairs", description="Test questions")
# client.create_examples(inputs={"question": "What is RAG?"}, outputs={"answer": "Retrieval-Augmented Generation"}, dataset_id=dataset.id)
# Then run evaluation.